# Ejecución de inferencia y harness

## 1. Clonar repositorio

In [1]:
# Clonar el repo (codigo versionado, incluye config.yaml y fuentes.yaml)
!git clone --branch refactor-harness --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus

Cloning into '/content/corpus'...
remote: Enumerating objects: 617, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 617 (delta 85), reused 88 (delta 55), pack-reused 454 (from 1)
Receiving objects: 100% (617/617), 13.10 MiB | 18.91 MiB/s, done.
Resolving deltas: 100% (231/231), done.


In [32]:
%cd /content/corpus
%ls

/content/corpus
M1/  M2/  README.md


## 2. Instalar dependencias

In [13]:
!pip install -r M2/run_inference/requirements.txt
!pip install -r M2/harness/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## 3. Configuración de paths y checks de recursos necesitados

In [14]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

checks = {
    "Adaptador LoRA": PROJECT_ROOT / "M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors",
    "Gold set": PROJECT_ROOT / "M2/eval_harness/gold_examples_20.jsonl",
}
for nombre, path in checks.items():
    print(f"{nombre}: {'OK' if path.exists() else 'FALTA'} -- {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adaptador LoRA: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors
Gold set: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples_20.jsonl


In [15]:
from pathlib import Path
from google.colab import userdata

with open("/content/corpus/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={userdata.get('GROQ_API_KEY')}\n")
print(".env generado (no versionado, solo vive en este runtime)")

.env generado (no versionado, solo vive en este runtime)


## 4. Correr inferencia

In [34]:
%cd M2/run_inference/

/content/corpus/M2/run_inference


In [35]:
!python run_inference.py

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cpu for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading weights: 100% 197/197 [00:00<00:00, 9970.65it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing

## 5. Correr harness

In [40]:
%cd ..

/content/corpus/M2
common.py                              examples/        __pycache__/
dimension1b_similitud_semantica.ipynb  harness/         README.md
dimension1_exact_match.ipynb           harness.ipynb    run_inference/
ejecucion/                             llm_judge.ipynb


In [42]:
%cd harness
%ls

/content/corpus/M2/harness
cached_system.py  harness.py              metrics_judge.py     requirements.txt
config.yaml       length_bias_pairs.json  metrics_semantic.py  run_harness.py
gold_loader.py    metrics_exact.py        __pycache__/         scorecard.py


In [47]:
!python run_harness.py

Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples_20.jsonl | existe: True
Predicciones: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/predictions/encoder_solo.json | existe: True
Output: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs
Gold set cargado: 20 ejemplos
Formato valido
Primer ejemplo:
  text[:120]: Varón soltero de 37 años de edad, trabajador del campo, que es enviado a nuestro Servicio desde otro hospital por no ten ...
  entities_gold: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis', 'carcinoma verrugoso de Buscke-Lowenstein', 'condiloma acuminado gigante de Buschke-Lowenstein', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópico', 'infiltraba igualmente los tejidos pubia